# Puzzle

https://thefiddler.substack.com/p/can-you-pile-the-primes

### This Week’s Fiddler

From Dean Ballard comes a premier puzzle of primes:

Suppose you want to make two groups with equal sums using the first N2 prime numbers. What is the smallest value of N2 for which you can do this?

The answer is three! (Clearly, that wasn’t actually the puzzle.)

The first three primes are 2, 3, and 5, and you can split them up into two sets: {2, 3} and {5}. Sure enough, 2 + 3 = 5.

Your puzzle involves making three groups with equal sums using the first N3 prime numbers. What is the smallest value of N3 for which you can do this?

### This Week’s Extra Credit

Now you want to make six groups with equal sums using the first N6 prime numbers. What is the smallest value of N6 for which you can do this?

# Fiddler Solution

I think it is possible to do this by hand.

The first few primes are:

> P = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]

The corresponding cumulative sums are:

> S = [2, 5, 10, 17, 28, 41, 58, 77, 100, 129]

129 is the first of these numbers that is divisible by 3. 

> 129/3 = 43.

Considering these numbers, by trial and error, I found at least one partition of P = P1 + P2 + P3.

> P1 = [29, 11, 3]

> P2 = [23, 7, 13]

> P3 = [2, 5, 17, 19]

So this works, and N3 = 10.

# Extra Credit Solution

The main thing I learned from the magic squares puzzle ( https://github.com/sgauria/Fiddler/blob/main/Fiddler_2025_12_19.ipynb ) was that libraries matter, esp for NP-hard problems like this one. ( This puzzle is a case of https://en.wikipedia.org/wiki/Multiway_number_partitioning ). So, this time around, I am jumping straight to that instead of trying to roll my own code.

ORtools has a very very related page: https://developers.google.com/optimization/pack/multiple_knapsack , which I am adapting below.

In [1]:
# https://developers.google.com/optimization/pack/multiple_knapsack
# """Solve a multiple knapsack problem using a MIP solver."""
from ortools.linear_solver import pywraplp

def multiple_knapsack_mip(weights, values, capacities, debug=False):
    data = {}
    data["weights"] = weights
    data["values"] = values
    assert len(data["weights"]) == len(data["values"])
    data["num_items"] = len(data["weights"])
    data["all_items"] = range(data["num_items"])

    data["bin_capacities"] = capacities
    data["num_bins"] = len(data["bin_capacities"])
    data["all_bins"] = range(data["num_bins"])

    # Create the mip solver with the SCIP backend.
    solver = pywraplp.Solver.CreateSolver("SCIP")
    if solver is None:
        print("SCIP solver unavailable.")
        return

    # Variables.
    # x[i, b] = 1 if item i is packed in bin b.
    x = {}
    for i in data["all_items"]:
        for b in data["all_bins"]:
            x[i, b] = solver.BoolVar(f"x_{i}_{b}")

    # Constraints.
    # Each item is assigned to at most one bin.
    for i in data["all_items"]:
        solver.Add(sum(x[i, b] for b in data["all_bins"]) <= 1)

    # The amount packed in each bin cannot exceed its capacity.
    for b in data["all_bins"]:
        solver.Add(
            sum(x[i, b] * data["weights"][i] for i in data["all_items"])
            <= data["bin_capacities"][b]
        )

    # Objective.
    # Maximize total value of packed items.
    objective = solver.Objective()
    for i in data["all_items"]:
        for b in data["all_bins"]:
            objective.SetCoefficient(x[i, b], data["values"][i])
    objective.SetMaximization()

    print(f"Solving with {solver.SolverVersion()}")
    status = solver.Solve()

    if status == pywraplp.Solver.OPTIMAL:
        if (debug):
            print(f"Total packed value: {objective.Value()}")
        total_weight = 0
        bws = []
        for b in data["all_bins"]:
            if (debug):
                print(f"Bin {b}")
            bw = []
            bin_weight = 0
            bin_value = 0
            for i in data["all_items"]:
                if x[i, b].solution_value() > 0:
                    if (debug):
                        print(
                            f"Item {i} weight: {data['weights'][i]} value:"
                            f" {data['values'][i]}"
                        )
                    bin_weight += data["weights"][i]
                    bin_value += data["values"][i]
                    bw.append(data["weights"][i])
            if (debug):
                print(f"Packed bin weight: {bin_weight}")
                print(f"Packed bin value: {bin_value}\n")
            total_weight += bin_weight
            bws.append(bw)
        if (debug):
            print(f"Total packed weight: {total_weight}")
        return total_weight, bws
    else:
        if (debug):
             print("The problem does not have an optimal solution.")
        return None, None

#multiple_knapsack_mip([48, 30, 42, 36, 36, 48, 42, 42, 36, 24, 30, 30, 42, 36, 36], [10, 30, 25, 50, 35, 30, 15, 40, 30, 35, 45, 10, 20, 30, 25], [100,100,100,100,100], debug=True)

In [2]:
# https://developers.google.com/optimization/pack/multiple_knapsack
# """Solves a multiple knapsack problem using the CP-SAT solver."""
from ortools.sat.python import cp_model

def multiple_knapsack_cp(weights, values, capacities, debug=False):
    data = {}
    data["weights"] = weights
    data["values"] = values
    assert len(data["weights"]) == len(data["values"])
    num_items = len(data["weights"])
    all_items = range(num_items)

    data["bin_capacities"] = capacities
    num_bins = len(data["bin_capacities"])
    all_bins = range(num_bins)

    model = cp_model.CpModel()

    # Variables.
    # x[i, b] = 1 if item i is packed in bin b.
    x = {}
    for i in all_items:
        for b in all_bins:
            x[i, b] = model.new_bool_var(f"x_{i}_{b}")

    # Constraints.
    # Each item is assigned to at most one bin.
    for i in all_items:
        model.add_at_most_one(x[i, b] for b in all_bins)

    # The amount packed in each bin cannot exceed its capacity.
    for b in all_bins:
        model.add(
            sum(x[i, b] * data["weights"][i] for i in all_items)
            <= data["bin_capacities"][b]
        )

    # Objective.
    # maximize total value of packed items.
    objective = []
    for i in all_items:
        for b in all_bins:
            objective.append(cp_model.LinearExpr.term(x[i, b], data["values"][i]))
    model.maximize(cp_model.LinearExpr.sum(objective))

    solver = cp_model.CpSolver()
    status = solver.solve(model)

    if status == cp_model.OPTIMAL:
        if (debug):
            print(f"Total packed value: {solver.objective_value}")
        total_weight = 0
        bws = []
        for b in all_bins:
            if (debug):
                print(f"Bin {b}")
            bin_weight = 0
            bin_value = 0
            bs = []
            for i in all_items:
                if solver.value(x[i, b]) > 0:
                    if (debug):
                        print(
                            f'Item:{i} weight:{data["weights"][i]} value:{data["values"][i]}'
                        )
                    bin_weight += data["weights"][i]
                    bin_value += data["values"][i]
                    bs.append(data["weights"][i])
            if (debug):
                print(f"Packed bin weight: {bin_weight}")
                print(f"Packed bin value: {bin_value}\n")
            total_weight += bin_weight
            bws.append(bs)
        if (debug):
            print(f"Total packed weight: {total_weight}")
        return total_weight, bws
    else:
        if (debug):
            print("The problem does not have an optimal solution.")
        return None, None
    
#multiple_knapsack_cp([48, 30, 42, 36, 36, 48, 42, 42, 36, 24, 30, 30, 42, 36, 36], [10, 30, 25, 50, 35, 30, 15, 40, 30, 35, 45, 10, 20, 30, 25], [100,100,100,100,100], debug=True)

Now I actually write some code myself. :)

In [3]:
def multiway_number_partitioning(numbers, num_bins, knapack_fn=multiple_knapsack_cp, debug=False):
    sum_numbers = sum(numbers)
    weights = numbers
    values = [1] * len(numbers)
    bin_capacities = [sum_numbers // num_bins] * num_bins
    total_used, bins = knapack_fn(weights, values, bin_capacities, debug=debug)
    return total_used, bins

multiway_number_partitioning([2,3,5], 2)

(10, [[2, 3], [5]])

In [4]:
def primes_up_to(n):
    """Returns a list of primes up to n."""
    sieve = [True] * (n + 1)
    sieve[0] = sieve[1] = False
    for i in range(2, int(n**0.5) + 1):
        if sieve[i]:
            for j in range(i * i, n + 1, i):
                sieve[j] = False
    return [i for i in range(n + 1) if sieve[i]]

primes_up_to(23)

[2, 3, 5, 7, 11, 13, 17, 19, 23]

In [5]:
def find_N(prime_limit, num_bins_limit):
    primes = primes_up_to(prime_limit)
    prime_sums = [0]
    sum = 0
    for i in range(len(primes)):
        sum += primes[i]
        prime_sums.append(sum)
    
    for num_bins in range(1, num_bins_limit + 1):
        #print(f"Trying with {num_bins} bins")
        for i in range(1, len(primes) + 1):
            #print(f"Trying with first {i} primes {primes[0:i]}, sum: {prime_sums[i]}")
            if prime_sums[i] % num_bins == 0:
                total_used, bins = multiway_number_partitioning(primes[0:i], num_bins)
                if total_used is not None and total_used == prime_sums[i]:
                    print(f"Found N_{num_bins}: {i} primes can be partitioned into {num_bins} bins")
                    print(f"   Bins: {bins}")
                    break

find_N(1000, 9)

Found N_1: 1 primes can be partitioned into 1 bins
   Bins: [[2]]
Found N_2: 3 primes can be partitioned into 2 bins
   Bins: [[2, 3], [5]]
Found N_3: 10 primes can be partitioned into 3 bins
   Bins: [[11, 13, 19], [3, 17, 23], [2, 5, 7, 29]]
Found N_4: 11 primes can be partitioned into 4 bins
   Bins: [[17, 23], [11, 29], [3, 5, 13, 19], [2, 7, 31]]
Found N_5: 17 primes can be partitioned into 5 bins
   Bins: [[29, 59], [7, 11, 17, 53], [41, 47], [3, 19, 23, 43], [2, 5, 13, 31, 37]]
Found N_6: 57 primes can be partitioned into 6 bins
   Bins: [[23, 61, 193, 197, 199, 233, 239], [11, 17, 19, 43, 53, 89, 127, 151, 167, 227, 241], [31, 41, 59, 73, 103, 179, 191, 211, 257], [5, 47, 163, 181, 229, 251, 269], [7, 13, 79, 83, 97, 101, 107, 109, 137, 149, 263], [2, 3, 29, 37, 67, 71, 113, 131, 139, 157, 173, 223]]
Found N_7: 22 primes can be partitioned into 7 bins
   Bins: [[17, 23, 73], [29, 31, 53], [11, 41, 61], [7, 47, 59], [5, 37, 71], [3, 43, 67], [2, 13, 19, 79]]
Found N_8: 29 primes

*cp solver took  1m 34.5s. mip solver took over 6m, and hadn't finished, and has this distracting "Solving with SCIP 9.2.2 ... " printout.  So, cp solver is clearly better. :)*

Extra Credit Answer is 57 primes.

Some OEIS searching shows that Dean has already filed this sequence: https://oeis.org/A390531 , which helps confirm these answers. :)